# Day 6 — Decision Tree + Random Forest

**Goal:** train tree models, compare to **logistic regression** on the same test split, plot **feature importance**.

> **Read first:** [CONCEPTS.md](./CONCEPTS.md)


## Setup

Run with cwd `notebooks/day06/`. Tree helpers: `src/tree_models.py`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "dataset").is_dir() and (p / "requirements.txt").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification import fit_baseline
from src.preprocessing import load_cleaned_table, prepare_train_test_bundle, save_artifacts
from src.tree_models import (
    comparison_rows,
    evaluate_named_models,
    fit_tree_models,
    plot_feature_importance,
    random_forest_importance_df,
    save_day06_artifacts,
)

sns.set_theme(style="whitegrid", context="notebook")

## 1. Same Day 3 features and split (`random_state=42`)

In [ ]:
df = load_cleaned_table(PROJECT_ROOT)
bundle = prepare_train_test_bundle(df, test_size=0.2, random_state=42)
save_artifacts(bundle, PROJECT_ROOT)

X_train, X_test = bundle["X_train"], bundle["X_test"]
y_train, y_test = bundle["y_train"], bundle["y_test"]
feature_names = bundle["feature_names"]
class_names = bundle["career_classes"]

## 2. Train logistic baseline + tree models

- **Logistic regression** — Day 5 reference
- **Decision tree** — `max_depth=12` to limit extreme overfitting
- **Random forest** — 200 trees, bagging

In [ ]:
logistic = fit_baseline(X_train, y_train)
trees = fit_tree_models(X_train, y_train)

models = {
    "logistic_regression": logistic,
    "decision_tree": trees["decision_tree"],
    "random_forest": trees["random_forest"],
}

## 3. Test-set comparison table

Report these numbers in your report (not slide placeholders).

In [ ]:
evals = evaluate_named_models(models, X_test, y_test, class_names)
comparison = pd.DataFrame(comparison_rows(evals))
comparison.sort_values("f1_macro", ascending=False)

## 4. Random Forest feature importance

In [ ]:
importance_df = random_forest_importance_df(trees["random_forest"], feature_names, top_n=15)
importance_df

In [ ]:
fig = plot_feature_importance(importance_df)
plt.show()

## 5. Save comparison, plots, and RF model

In [ ]:
paths = save_day06_artifacts(
    trees["random_forest"],
    importance_df,
    comparison,
    evals,
    class_names,
    PROJECT_ROOT,
)
paths

## Checkpoint

1. Did the single tree beat the forest on **test**? Why might that happen?
2. Name two top features from importance — do they match your domain intuition?